# Draft-Revise: Systematic Study

Each experiment below shows **exactly what parameters changed** vs the CTM baseline.

## Experimental Matrix

| group | what changes | tasks | seeds | runs |
|---|---|---|---|---|
| **Main** | +revise mechanism at best config | 4 | 5 | 20 |
| **Sweep** | revise_weight × corrupt_prob grid | 2 | 3 | 72 |
| **Ablation** | remove/alter one component at a time | 2 | 3 | 30 |
| **Total** | | | | **122** |

In [ ]:
import sys; sys.path.insert(0, '.'); sys.path.insert(0, '..')
from exp_runner import *
import matplotlib.pyplot as plt
%matplotlib inline

TASKS = ['sort', 'cifar10', 'mazes', 'parity']
SEEDS_MAIN = [0, 1, 2, 3, 4]
SEEDS_SWEEP = [0, 1, 2]

In [ ]:
# Show baseline configs so readers can see what we're modifying
for task in TASKS:
    module, cfg = BASE_CONFIGS[task]
    print(f'{task:10s} d_model={cfg.get("d_model")}, iters={cfg.get("iterations")}, '
          f'lr={cfg.get("lr")}, train_iters={cfg.get("training_iterations")}')

## Prior Results (st10 from initial sweep)

In [ ]:
df_prior = load_prior()
if df_prior is not None:
    print(summary_stats(df_prior[df_prior.stage == 'st10']))
    plot_prior_bar(df_prior, ['cifar10','mazes','parity','sort'],
                   'st10', 'revise', 'Prior: revise vs baseline',
                   'figures/01_prior_bar.png')
else:
    print('Prior data not found.')

## Group 1 — Main Experiment (20 runs)

**Goal**: Validate the best revise config (w=0.1, cp=0.15) with 5 seeds.

**Δ from baseline** (same for all tasks):
```
+ draft_mode         = 'revise'     # enable draft-then-revise mechanism
+ draft_revise_weight = 0.1          # weight of revision loss
+ draft_corrupt_prob  = 0.15         # 15% of tokens get corrupted in draft
+ draft_block_size   = 2            # revise 2-token blocks
```

In [ ]:
exps_main = []
for task in TASKS:
    module, base = BASE_CONFIGS[task]
    for s in SEEDS_MAIN:
        exps_main.append(Experiment(
            name=f'{task}_revise_w0.1_cp0.15_s{s}',
            task=task, module=module,
            config={**base, 'seed': s,
                    'draft_mode': 'revise',            # + enable revise
                    'draft_revise_weight': 0.1,         # + revision loss weight
                    'draft_corrupt_prob': 0.15,         # + 15% token corruption
                    'draft_block_size': 2}))            # + 2-token revise blocks
print(f'{len(exps_main)} experiments')
for e in exps_main[:4]:
    print(f'  {e.name}')

## Group 2 — Hyperparameter Sweep (72 runs)

**Goal**: Map the revise_weight × corrupt_prob landscape on sort + mazes.

**Sweep grid**:
- `draft_revise_weight` ∈ {0.05, 0.1, 0.2, 0.3}
- `draft_corrupt_prob` ∈ {0.05, 0.15, 0.30}
- 3 seeds × 2 tasks = 72 total

In [ ]:
SWEEP_W = [0.05, 0.1, 0.2, 0.3]
SWEEP_CP = [0.05, 0.15, 0.30]
exps_sweep = []
for task in ['sort', 'mazes']:
    module, base = BASE_CONFIGS[task]
    for w in SWEEP_W:
        for cp in SWEEP_CP:
            for s in SEEDS_SWEEP:
                exps_sweep.append(Experiment(
                    name=f'{task}_swp_w{str(w).replace(".","p")}_cp{str(cp).replace(".","p")}_s{s}',
                    task=task, module=module,
                    config={**base, 'seed': s,
                            'draft_mode': 'revise',
                            'draft_revise_weight': w,        # SWEEP: 0.05~0.3
                            'draft_corrupt_prob': cp,        # SWEEP: 0.05~0.30
                            'draft_block_size': 2}))
print(f'{len(exps_sweep)} experiments')
print(f'Grid: {len(SWEEP_W)} weights x {len(SWEEP_CP)} corrupts x {len(SEEDS_SWEEP)} seeds x 2 tasks')

## Group 3 — Ablation Study (30 runs)

Each variant removes or alters **one** component. Same base: w=0.1, cp=0.15.

| variant | what changes | question |
|---|---|---|
| `full` | nothing (reference) | baseline of ablation |
| `no_noise` | cp=0.0 | Is corruption essential? |
| `no_revise_loss` | w=0.0 | Is the revision loss needed, or just the draft pass? |
| `block1` | block_size=1 | Does block size matter? |
| `block3` | block_size=3 | Does block size matter? |

In [ ]:
ABLATIONS = [
    # (variant_name, overrides_dict, description)
    ('full',            {},                                              'reference'),
    ('no_noise',        {'draft_corrupt_prob': 0.0},                    'corruption OFF'),
    ('no_revise_loss',  {'draft_revise_weight': 0.0},                   'revision loss OFF'),
    ('block1',          {'draft_block_size': 1},                        '1-token blocks'),
    ('block3',          {'draft_block_size': 3},                        '3-token blocks'),
]
exps_ablation = []
for task in ['sort', 'mazes']:
    module, base = BASE_CONFIGS[task]
    for variant, overrides, desc in ABLATIONS:
        for s in SEEDS_SWEEP:
            cfg = {**base, 'seed': s,
                   'draft_mode': 'revise',
                   'draft_revise_weight': 0.1,
                   'draft_corrupt_prob': 0.15,
                   'draft_block_size': 2}
            cfg.update(overrides)  # apply ablation override
            exps_ablation.append(Experiment(
                name=f'{task}_abl_{variant}_s{s}',
                task=task, module=module, config=cfg))
print(f'{len(exps_ablation)} experiments')
for v, _, d in ABLATIONS:
    print(f'  {v:20s} ({d})')

## Run All Experiments

In [ ]:
exps = exps_main + exps_sweep + exps_ablation
print(f'Total: {len(exps)} experiments')
run_all(exps, gpus=8, log_root='logs/deep/01_revise', dry_run=True)

In [ ]:
done, failed = run_all(exps, gpus=8, log_root='logs/deep/01_revise')

In [ ]:
status('logs/deep/01_revise')

## Analysis

In [ ]:
df = collect('logs/deep/01_revise')
if df.empty:
    print('No results yet.')
else:
    df_main = df[df.name.str.contains('revise_w0.1_cp0.15') & ~df.name.str.contains('swp|abl')]
    if not df_main.empty:
        print(df_main[['name','task','best_acc','delta']].to_string(index=False))
        plot_delta_bars(df_main, 'Main: revise vs baseline (5 seeds)', 'figures/01_main_delta.png')
        print(significance_test(df_main).to_string(index=False))

In [ ]:
if not df.empty:
    import re
    df_sw = df[df.name.str.contains('swp_')].copy()
    if not df_sw.empty:
        df_sw['weight'] = df_sw['name'].str.extract(r'w([0-9]+p?[0-9]*)_cp')[0].str.replace('p','.').astype(float)
        plot_sweep_heatmap(df_sw, 'weight', 'task',
                          'Revise: weight x task delta (pp)', 'figures/01_sweep_heatmap.png')
        plot_sweep_curve(df_sw, 'weight', 'Weight sweep', 'figures/01_sweep_curve.png')

In [ ]:
if not df.empty:
    df_abl = df[df.name.str.contains('abl_')].copy()
    if not df_abl.empty:
        import re
        df_abl['variant'] = df_abl['name'].str.extract(r'abl_([a-z_]+)_s')[0]
        plot_ablation_bars(df_abl, 'variant',
                          'Ablation: component contributions', 'figures/01_ablation.png')
        print(summary_stats(df_abl, groupby=('task','variant')))